## Chroma CRUD OPERATIONS

Walks through create, read, update and delete with a local Chroma Vector Store

In [6]:
collection_name="demo"
project_root = Path.cwd()
persists_directory = project_root / "chroma_langchain_db"

In [10]:
vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
    persist_directory=str(persists_directory)
)
print("vector store is ready")

vector store is ready


In [14]:
# create and insert examle documents
# Keep the raw sample data separate from the Document objects so it is easier to read.
document_examples = [
    {
        "topic": "AI",
        "doc_number": 1,
        "text": "Artificial intelligence helps machines perform tasks that usually need human reasoning.",
    },
    {
        "topic": "AI",
        "doc_number": 2,
        "text": "AI systems can analyze patterns in data to support predictions and automation.",
    },
    {
        "topic": "AI",
        "doc_number": 3,
        "text": "Responsible AI development includes fairness, transparency, and safety checks.",
    },
    {
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG combines retrieval with generation so the model can answer using external knowledge.",
    },
    {
        "topic": "RAG",
        "doc_number": 5,
        "text": "A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.",
    },
    {
        "topic": "RAG",
        "doc_number": 6,
        "text": "Vector stores are important in RAG because they make semantic search over embedded documents possible.",
    },
    {
        "topic": "LLM",
        "doc_number": 7,
        "text": "LLMs generate text by predicting likely next tokens from patterns learned during training.",
    },
    {
        "topic": "LLM",
        "doc_number": 8,
        "text": "Prompt design can improve how clearly an LLM follows instructions and returns useful answers.",
    },
    {
        "topic": "Cricket",
        "doc_number": 9,
        "text": "Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.",
    },
    {
        "topic": "Cricket",
        "doc_number": 10,
        "text": "A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.",
    },
]

print(f"Prepared {len(document_examples)} document examples.")

Prepared 10 document examples.


In [16]:
# Convert the sample data into LangChain Document objects.
documents = [
    Document(
        id=str(uuid4()),
        page_content=item["text"],
        metadata={"topic": item["topic"], "doc_number": item["doc_number"]},
    )
    for item in document_examples
]

print_documents("Dummy documents prepared:", documents)

Dummy documents prepared:
1: id=0e756956-1edf-4fd7-a341-1102e1837704    topic=AI    doc_number=1
    content=Artificial intelligence helps machines perform tasks that usually need human reasoning.
2: id=7a3abc5c-1ee9-4421-a55e-ceab293aa868    topic=AI    doc_number=2
    content=AI systems can analyze patterns in data to support predictions and automation.
3: id=b9195900-9a03-44ff-ac76-e565db4724c6    topic=AI    doc_number=3
    content=Responsible AI development includes fairness, transparency, and safety checks.
4: id=5cea8c46-195c-4a39-a377-d6b2c7204ec9    topic=RAG    doc_number=4
    content=RAG combines retrieval with generation so the model can answer using external knowledge.
5: id=bf41bd6f-0756-4b9a-9cbf-3455d9a83a7b    topic=RAG    doc_number=5
    content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.
6: id=07d51640-83dd-4b5d-ae21-f790d7944e39    topic=RAG    doc_number=6
    content=Vector stores are important in RAG beca

## Read the Stored Data Back

In [22]:
print(raw_records["embeddings"].shape)
print(f"Total records in collection: {len(raw_records['ids'])}")
print("First three ids from get():")
for doc_id in raw_records["ids"][:3]:
    print(doc_id)

(10, 768)
Total records in collection: 10
First three ids from get():
0e756956-1edf-4fd7-a341-1102e1837704
7a3abc5c-1ee9-4421-a55e-ceab293aa868
b9195900-9a03-44ff-ac76-e565db4724c6


## Run a Similarity Search

In [25]:
search_results = vector_store.similarity_search(query, k=3)
print(f"Query: {query}\n")
print_documents("Similarity search results:", search_results)

Query: How does RAG help an LLM answer questions using outside knowledge?

Similarity search results:
1: id=5cea8c46-195c-4a39-a377-d6b2c7204ec9    topic=RAG    doc_number=4
    content=RAG combines retrieval with generation so the model can answer using external knowledge.
2: id=e57298a2-cf9e-486d-8aa9-d29b9346f9f0    topic=LLM    doc_number=8
    content=Prompt design can improve how clearly an LLM follows instructions and returns useful answers.
3: id=bf41bd6f-0756-4b9a-9cbf-3455d9a83a7b    topic=RAG    doc_number=5
    content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.



## Update Existing Documents

In [28]:
# Keep the replacement text separate so the update step stays easy to follow.
updated_examples = [
    {
        "id": ids_to_update[0],
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG improves answer quality by retrieving relevant context before the language model generates a response.",
    },
    {
        "id": ids_to_update[1],
        "topic": "LLM",
        "doc_number": 8,
        "text": "Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.",
    },
]

updated_documents = [
    Document(
        id=item["id"],
        page_content=item["text"],
        metadata={"topic": item["topic"], "doc_number": item["doc_number"]},
    )
    for item in updated_examples
]

print_documents("Updated document content:", updated_documents)

Updated document content:
1: id=5cea8c46-195c-4a39-a377-d6b2c7204ec9    topic=RAG    doc_number=4
    content=RAG improves answer quality by retrieving relevant context before the language model generates a response.
2: id=e57298a2-cf9e-486d-8aa9-d29b9346f9f0    topic=LLM    doc_number=8
    content=Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.



In [30]:
updated_raw_records = vector_store.get(ids=ids_to_update)

In [32]:
updated_query = "How can retrieved context improve an LLM response in RAG?"
updated_query

'How can retrieved context improve an LLM response in RAG?'

## Delete Documents

In [37]:
remaining_records = vector_store.get()
remaining_ids = remaining_records["ids"]
print([doc.metadata["topic"] for doc in documents if doc.id in remaining_ids])

['AI', 'AI', 'AI', 'RAG', 'RAG', 'RAG', 'LLM', 'LLM']
